# 1- Import packages

In [1]:
import time 
from collections import deque, namedtuple 

import gym 
import numpy as np 
import PIL.Image
import tensorflow as tf 

from pyvirtualdisplay import Display
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense, Input 
from tensorflow.keras.losses import MSE 
from tensorflow.keras.optimizers import Adam

# 2- Hyperparameters 

In [2]:
MEMORY_SIZE = 100_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-3              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps

# 3 - The Lunar Lander Environment

The goal of the Lunar Lander environment is to land the lunar lander safely on the landing pad on the surface of the moon. The landing pad is designed by two flag poles and its center is at coordinates `(0, 0)`, but the lander is also allowed to land outside of the pad. The lander starts at the top center of the environment with a random initial force applied to its center of mass and has infinite fuel. The environment is considered solved if you get `200` points. 

In [4]:
!pip install "gymnasium[box2d]"

In [5]:
import gymnasium as gym 
from PIL import Image

# Initialize the environment with correct render_mode
env = gym.make("LunarLander-v3", render_mode="rgb_array")

# Reset the environment
obs, info = env.reset()

# Render the initial frame
frame = env.render()

# Convert to an image
img = Image.fromarray(frame)
img.show()

In [7]:
state_size = env.observation_space.shape 
num_actions = env.action_space.n 

print ('State Shape:', state_size) 
print('Number of actions:', num_actions) 

State Shape: (8,)
Number of actions: 4


In [8]:
current_state = env.reset() 

Once the environment is reset, the agent can start taking actions in the environment by using the `.step()` method. Note that the agent can only take on action per time step.

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

In [9]:
action = 0 
next_state, reward, terminated, truncated, info = env.step(action) 
done = terminated or truncated 

current_state = next_state

# 4 - Deep Q-learning 

In [11]:
q_network = Sequential([
    tf.keras.layers.Input(shape = state_size), 
    tf.keras.layers.Dense(64, activation = 'relu'), 
    tf.keras.layers.Dense(64, activation = 'relu'), 
    tf.keras.layers.Dense(num_actions, activation = 'linear'),
]) 

target_q_network = Sequential([
    tf.keras.layers.Input(shape = state_size), 
    tf.keras.layers.Dense(64, activation = 'relu'), 
    tf.keras.layers.Dense(64, activation = 'relu'), 
    tf.keras.layers.Dense(num_actions, activation = 'linear'),
]) 

optimizer = tf.keras.optimizers.Adam(learning_rate = ALPHA) 

In [13]:
experienced = namedtuple("Experience", field_names=["state", "action", "reward", "next_state"])

$$
\begin{equation}
    y_j =
    \begin{cases}
      R_j & \text{if episode terminates at step  } j+1\\
      R_j + \gamma \max_{a'}\hat{Q}(s_{j+1},a') & \text{otherwise}\\
    \end{cases}       
\end{equation}
$$

Here are a couple of things to note: 

* The `compute_loss` functions takes in a mini-batch of experienced tuples. This mini-batch of experience tuples is unpacked to extra the `states`, `actions`, `rewards`, `next_states` and `done_vals`. You should keep in mind that these variables are Tensorflow Tensors whose size will depend on the mini-batch size. For example, if the mini-batch is `64` then both `rewards` and `done_vals` will be TensorFlow Tensors with `64` elements. 

In [ ]:
def compute_loss(experiences, gamma, q_network, target_q_network): 
    
    states, actions, rewards, next_states, done_vals = experiences 
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)

    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))

    q_values = q_network(states) 
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))

    loss = MSE(y_targets, q_values) 
    return loss 

In [ ]:
def update_target_network(q_network, target_q_network): 
    for target_weights, q_net_weights in zip(
        target_q_network.weights, q_network.weights
    ): 
        target_weights.assign(TAU * q_net_weights + (1.0 - TAU) * target_weights)

In [ ]:
@tf.function 
def agent_learn(experiences, gamma): 
    with tf.GradientTape() as tape: 
        loss = compute_loss(experiences, gamma, q_network, target_q_network)

    gradients = tape.gradient(loss, q_network.trainable_variables) 
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

    update_target_network(q_network, target_q_network) 

In [ ]:
def check_update_condition(t, num_steps_upd, memory_buffer): 
    if (t + 1) % num_steps_upd == 0 and len(memory_buffer) > MINIBATCH_SIZE: 
        return True 
    else: 
        return False 

In [ ]:
def get_action(q_values, epsilon = 0.0): 
    if random.random() > epsilon: 
        return np.argmax(q_values.numpy()[0])
    return 
        return random.choice(np.arrange(4)) 

In [ ]:
def get_experiences(memory_buffer): 
    experiences = random.sample(memory_buffer, k = MINIBATCH_SIZE) 
    states = tf.convert_to_tensor(
        np.array([e.state for e in experience if e is not None], dtype = tf.float32)
    ) 
    actions = tf.convert_to_tensor(
        np.array([e.action for e in experiences if e is not None], dtype = tf.float32)
    )
        rewards = tf.convert_to_tensor(
        np.array([e.reward for e in experiences if e is not None]), dtype=tf.float32
    )
    next_states = tf.convert_to_tensor(
        np.array([e.next_state for e in experiences if e is not None]), dtype=tf.float32
    )
    done_vals = tf.convert_to_tensor(
        np.array([e.done for e in experiences if e is not None]).astype(np.uint8),
        dtype=tf.float32,
    )
    return (states, actions, rewards, next_states, done_vals) 

In [ ]:
import random 

In [ ]:
def get_new_eps(epsilon):
    """
    Updates the epsilon value for the ε-greedy policy.
    
    Gradually decreases the value of epsilon towards a minimum value (E_MIN) using the
    given ε-decay rate (E_DECAY).

    Args:
        epsilon (float):
            The current value of epsilon.

    Returns:
       A float with the updated value of epsilon.
    """

    return max(E_MIN, E_DECAY * epsilon)

In [ ]:
start = time.time() 
MINIBATCH_SIZE = 64
TAU = 1e-3
E_DECAY = 0.995  # ε-decay rate for the ε-greedy policy.
E_MIN = 0.01  # Minimum ε value for the ε-greedy policy.
num_episodes = 2000
max_num_timesteps = 1000

total_point_history = []
num_p_av = 100 
epsilon = 1.0 
memory_buffer = deque(maxlen = MEMORY_SIZE) 
target_q_network.set_weights(q_network.get_weights()) 

for i in range(num_episodes): 
    states, _ = env.reset() 
    total_points = 0 

    for t in range(max_num_timesteps): 
        state_qn = np>expand_dims(state, axis = 0) 
        q_values = q_network(state_qn) 
        action = get_action(q_values, epsilon) 

        next_statem reward, terminated, truncated, info = env.step(action) 
        done = terminated or truncated 

        memory_buffer.append(experience(state, action, reward, next_state, done))

        update = check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)

        if update: 
            experiences = get_experiences(memory_buffer) 

            agent_learn(experiences, GAMMA) 

        state = next_state.copy() 
        total_points += reward 

        if done: 
            break

    total_point_history.append(total_points)
    avg_latest_points = np.mean(total_point_history[-num_p_av:])
    epsilon = get_new_eps(epsilon) 

    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")

    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")

    # We will consider that the environment is solved if we get an
    # average of 200 points in the last 100 episodes.
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break
        
tot_time = time.time() - start

print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")